# N02 — Main Evaluation Activation Capture v1.0

**Purpose:** capture block-boundary residual states transiently and save the three preregistered N02 observables per text × model × Transformer block:

1. Length-adjusted block-update norm, `||h_(l+1) - h_l||_F / sqrt(T)`
2. Input–update cosine, `cos(h_l, h_(l+1)-h_l)`
3. Output/input norm ratio, `||h_(l+1)||_F / ||h_l||_F`

The complete hidden tensors are used in memory to compute these observables and are not persisted; the output stores per-text, per-block metrics and audit metadata. This is appropriate for N02's fixed endpoint set, not a raw-activation archive for later arbitrary analyses.

## Mandatory gate

This notebook independently verifies the exact frozen manifest bytes and text fingerprint, then **recomputes** structure, both tokenizer length bands, Python syntax, duplicate text, numeric-masked code-template duplication, and code-topic cues. It does not trust a cached `checks` object or a report's `human_content_review_complete` field alone.

The uploaded `N02_Main_Evaluation_Dataset_Design_v1_0claudeSonnet.ipynb` is a preflight notebook; it does not perform activation capture. Its final cells set `CONTENT_REVIEW_COMPLETE = True` directly and reuse some earlier checks, so its reported freeze is not, by itself, enough to authorize this confirmatory run.

**Do not change the gate flags merely to make the run start.** If any content checks fail, repair the manifest, rerun the audit, review changes, and use a new manifest hash before capturing activations.


In [ ]:
%pip -q install -U transformers pandas

## 1. Load the frozen manifest and report

Run this cell after placing the manifest and preflight report in `/content/n02_main_eval_preflight/`, or upload both files (or a ZIP containing them). The report's `frozen_for_confirmatory_analysis` flag and hashes must match the exact CSV bytes.


In [ ]:
from pathlib import Path
import os, json, hashlib, re, ast, math, gc, platform, sys, zipfile
from datetime import datetime, timezone
import pandas as pd
import torch
import transformers
from IPython.display import display

DEFAULT_DIR = Path("/content/n02_main_eval_preflight")
manifest_path = DEFAULT_DIR / "N02_Main_Evaluation_FROZEN_Manifest.csv"
report_path = DEFAULT_DIR / "N02_preflight_report.json"

if not (manifest_path.exists() and report_path.exists()):
    from google.colab import files
    print("Upload the frozen CSV and its matching N02_preflight_report.json, or a ZIP containing both.")
    uploaded = files.upload()
    upload_dir = Path("/content/n02_activation_input")
    upload_dir.mkdir(parents=True, exist_ok=True)
    for name, data in uploaded.items():
        target = upload_dir / Path(name).name
        target.write_bytes(data)
        if target.suffix.lower() == ".zip":
            with zipfile.ZipFile(target, "r") as zf:
                zf.extractall(upload_dir / target.stem)
    csv_candidates = list(upload_dir.rglob("*FROZEN*Manifest*.csv"))
    json_candidates = [p for p in upload_dir.rglob("*.json") if "preflight_report" in p.name.lower()]
    if len(csv_candidates) != 1 or len(json_candidates) != 1:
        raise FileNotFoundError(
            f"Need exactly one frozen-manifest CSV and one preflight report; found {csv_candidates}, {json_candidates}"
        )
    manifest_path, report_path = csv_candidates[0], json_candidates[0]

manifest_bytes = manifest_path.read_bytes()
manifest_file_sha256 = hashlib.sha256(manifest_bytes).hexdigest()
report = json.loads(report_path.read_text(encoding="utf-8"))
manifest = pd.read_csv(manifest_path, keep_default_na=False)

def sha256_text_fingerprint(frame):
    payload = "\n".join(
        f"{text_id}\n{text}"
        for text_id, text in zip(frame["text_id"].astype(str), frame["text"].astype(str))
    ).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()

text_fingerprint_sha256 = sha256_text_fingerprint(manifest)

print("Manifest:", manifest_path)
print("Report:", report_path)
print("Rows:", len(manifest), "| topics:", manifest["topic_id"].nunique() if "topic_id" in manifest else "MISSING")
print("Manifest file SHA-256:", manifest_file_sha256)
print("Text fingerprint SHA-256:", text_fingerprint_sha256)
print("Reported frozen:", report.get("frozen_for_confirmatory_analysis"))
print("Python:", sys.version.split()[0], "| PyTorch:", torch.__version__, "| Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))


## 2. Independent data-integrity recheck

This cell recomputes validations from the **current uploaded manifest**. It treats repeated code templates after numeric masking and missing topic cues as hard stops pending review, since those issues were previously identified in this project. It also recalculates token counts rather than trusting cached count columns.


In [ ]:
EXPECTED_STYLES = {
    "general_prose", "question_dialogue", "academic", "code_plus_natural_language"
}
LENGTH_BANDS = {"short": (16, 64), "medium": (65, 128), "long": (129, 256)}
TOKENIZER_IDS = {"gpt2": "gpt2", "pythia-160m": "EleutherAI/pythia-160m"}
REQUIRED = {"topic_id", "topic_domain", "length_tier", "style", "text_id", "text", "pilot_excluded"}
missing_columns = REQUIRED - set(manifest.columns)

base_checks = {
    "report_says_frozen": report.get("frozen_for_confirmatory_analysis") is True,
    "manifest_file_hash_matches_report": report.get("frozen_manifest_file_sha256") == manifest_file_sha256,
    "text_fingerprint_matches_report": report.get("candidate_text_manifest_sha256") == text_fingerprint_sha256,
    "report_says_400_rows": report.get("rows") == 400,
    "required_columns_present": len(missing_columns) == 0,
}

if missing_columns:
    raise ValueError(f"Manifest missing required columns: {sorted(missing_columns)}")

normalized = manifest["text"].astype(str).str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
base_checks.update({
    "rows_400": len(manifest) == 400,
    "topics_100": manifest["topic_id"].nunique() == 100,
    "text_id_unique": manifest["text_id"].is_unique,
    "four_styles_per_topic": bool((manifest.groupby("topic_id")["style"].nunique() == 4).all()),
    "one_row_per_topic_style": int(manifest[["topic_id", "style"]].duplicated().sum()) == 0,
    "one_length_tier_per_topic": bool((manifest.groupby("topic_id")["length_tier"].nunique() == 1).all()),
    "styles_exact": set(manifest["style"]) == EXPECTED_STYLES,
    "no_exact_normalized_duplicate_text": int(normalized.duplicated().sum()) == 0,
    "pilot_prompts_excluded": manifest["pilot_excluded"].astype(str).str.lower().isin(["true", "1"]).all(),
})

# Tokenizer lengths recalculated from actual current text.
tokenizers = {key: transformers.AutoTokenizer.from_pretrained(mid) for key, mid in TOKENIZER_IDS.items()}
for key, tokenizer in tokenizers.items():
    token_col = f"tokens_{key}_recomputed"
    band_col = f"in_band_{key}_recomputed"
    manifest[token_col] = [
        len(tokenizer.encode(str(text), add_special_tokens=True, truncation=False))
        for text in manifest["text"]
    ]
    manifest[band_col] = [
        LENGTH_BANDS[str(tier)][0] <= int(count) <= LENGTH_BANDS[str(tier)][1]
        for tier, count in zip(manifest["length_tier"], manifest[token_col])
    ]

# Check code syntax, duplicated templates after numeric masking, and topic cue presence.
code_df = manifest.loc[manifest["style"] == "code_plus_natural_language"].copy()
def python_syntax_error(text):
    try:
        ast.parse(str(text))
        return ""
    except SyntaxError as exc:
        return f"{exc.__class__.__name__} line {exc.lineno}: {exc.msg}"

def code_signature(text):
    text = str(text).lower().replace("\r\n", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\b\d+(?:\.\d+)?\b", "<num>", text)
    text = re.sub(r"\n+", "\n", text).strip()
    return text

def cue_present(row):
    domain_terms = [x for x in re.split(r"[_\s/\-]+", str(row["topic_domain"]).lower()) if len(x) >= 4]
    text = str(row["text"]).lower().replace("_", " ")
    # Require at least one domain component or the full normalized domain phrase.
    domain_phrase = str(row["topic_domain"]).lower().replace("_", " ")
    return (domain_phrase in text) or any(term in text for term in domain_terms)

code_df["syntax_error"] = code_df["text"].map(python_syntax_error)
code_df["topic_cue_present"] = code_df.apply(cue_present, axis=1)
code_df["numeric_masked_signature"] = code_df["text"].map(code_signature)
code_df["numeric_masked_duplicate"] = code_df["numeric_masked_signature"].duplicated(keep=False)

syntax_bad = code_df.loc[code_df["syntax_error"] != "", ["topic_id", "topic_domain", "text_id", "syntax_error", "text"]]
missing_cues = code_df.loc[~code_df["topic_cue_present"], ["topic_id", "topic_domain", "text_id", "text"]]
template_dupes = code_df.loc[code_df["numeric_masked_duplicate"], ["topic_id", "topic_domain", "text_id", "text"]].sort_values(["topic_domain", "topic_id"])
all_band_pass = bool(manifest[[f"in_band_{key}_recomputed" for key in TOKENIZER_IDS]].all(axis=None))

base_checks.update({
    "both_tokenizers_in_band": all_band_pass,
    "all_code_python_syntax_valid": len(syntax_bad) == 0,
    "all_code_rows_have_topic_cue": len(missing_cues) == 0,
    "no_numeric_masked_duplicate_code_templates": len(template_dupes) == 0,
})

print("INDEPENDENT CHECKS:")
display(pd.DataFrame([{"check": k, "pass": bool(v)} for k, v in base_checks.items()]))
print("Recomputed out-of-band rows:", int((~manifest[[f"in_band_{key}_recomputed" for key in TOKENIZER_IDS]].all(axis=1)).sum()))
print("Invalid Python code rows:", len(syntax_bad))
print("Code rows missing topic cues:", len(missing_cues))
print("Code rows in repeated numeric-masked templates:", len(template_dupes))

if len(syntax_bad):
    print("--- Syntax failures ---")
    display(syntax_bad)
if len(missing_cues):
    print("--- Missing topic cues ---")
    display(missing_cues)
if len(template_dupes):
    print("--- Numeric-masked template duplicate groups (must be reviewed/repaired) ---")
    display(template_dupes)

PRECHECK_PASSED = bool(all(base_checks.values()))
print("PRECHECK_PASSED =", PRECHECK_PASSED)


## 3. Human-review attestation gate

The older preflight notebook included a cell that sets `CONTENT_REVIEW_COMPLETE = True` directly. Do not inherit that flag. Set the attestation below to `True` only after reviewing the current frozen CSV, especially all code-mixed entries flagged above. This cell cannot itself prove semantic quality.


In [ ]:
# Set True only after reviewing the CURRENT exact manifest and its audit tables.
HUMAN_REVIEW_ATTESTED_FOR_THIS_HASH = False

CAPTURE_ALLOWED = bool(PRECHECK_PASSED and HUMAN_REVIEW_ATTESTED_FOR_THIS_HASH)
print("Precheck passed:", PRECHECK_PASSED)
print("Human review attested:", HUMAN_REVIEW_ATTESTED_FOR_THIS_HASH)
print("Activation capture allowed:", CAPTURE_ALLOWED)
if not CAPTURE_ALLOWED:
    print("STOP: do not load models or collect activations until all checks pass and the current manifest is reviewed.")


## 4. Block-boundary extraction functions

The first captured state is the tensor entering block 0; the following states are outputs of each Transformer block **before final LayerNorm**. Final LayerNorm is excluded from block-update deltas. Hooks are removed in a `finally` block. Results store scalar observables, not full hidden tensors.


In [ ]:
MODEL_IDS = {
    "gpt2": "gpt2",
    "pythia-160m": "EleutherAI/pythia-160m",
}
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float32
MAX_LENGTH = 512


def load_causal_lm(model_id, dtype=DTYPE):
    tokenizer = transformers.AutoTokenizer.from_pretrained(model_id)
    model = transformers.AutoModelForCausalLM.from_pretrained(
        model_id, torch_dtype=dtype, low_cpu_mem_usage=True
    )
    model.to(DEVICE)
    model.eval()
    return tokenizer, model


def resolve_blocks(model):
    if hasattr(model, "transformer") and hasattr(model.transformer, "h"):
        return model.transformer.h, "gpt2"
    if hasattr(model, "gpt_neox") and hasattr(model.gpt_neox, "layers"):
        return model.gpt_neox.layers, "gpt_neox_pythia"
    raise RuntimeError("Unsupported architecture; expected GPT-2 or GPT-NeoX/Pythia.")


def first_hidden_tensor(output):
    if isinstance(output, torch.Tensor):
        return output
    if isinstance(output, (tuple, list)):
        for item in output:
            if isinstance(item, torch.Tensor) and item.ndim == 3:
                return item
    raise TypeError(f"Could not find a [B,T,D] hidden tensor in output type {type(output)}")


@torch.inference_mode()
def extract_metrics_one_text(tokenizer, model, model_key, manifest_row):
    text = str(manifest_row["text"])
    encoded = tokenizer(
        text, return_tensors="pt", truncation=False, add_special_tokens=True
    )
    encoded = {k: v.to(DEVICE) for k, v in encoded.items()}
    n_tokens = int(encoded["input_ids"].shape[1])
    if n_tokens > MAX_LENGTH:
        raise ValueError(f"Runtime token count {n_tokens} exceeds MAX_LENGTH={MAX_LENGTH}")

    blocks, architecture = resolve_blocks(model)
    captured = {}
    handles = []

    def first_pre_hook(module, args):
        if not args or not isinstance(args[0], torch.Tensor):
            raise RuntimeError("Failed to capture input to block 0")
        captured[0] = args[0].detach().to("cpu", dtype=torch.float32).clone()[0]

    handles.append(blocks[0].register_forward_pre_hook(first_pre_hook))

    for i, block in enumerate(blocks):
        def block_post_hook(module, args, output, i=i):
            captured[i + 1] = first_hidden_tensor(output).detach().to("cpu", dtype=torch.float32).clone()[0]
        handles.append(block.register_forward_hook(block_post_hook))

    try:
        outputs = model(**encoded, use_cache=False)
    finally:
        for handle in handles:
            handle.remove()

    n_blocks = len(blocks)
    if len(captured) != n_blocks + 1 or any(i not in captured for i in range(n_blocks + 1)):
        raise RuntimeError(f"Expected {n_blocks + 1} block-boundary states; captured keys={sorted(captured)}")
    if not torch.isfinite(outputs.logits).all():
        raise FloatingPointError("Non-finite logits detected")

    states = [captured[i].to(dtype=torch.float64) for i in range(n_blocks + 1)]
    for i, h in enumerate(states):
        if not torch.isfinite(h).all():
            raise FloatingPointError(f"Non-finite hidden state {i}")

    rows = []
    for l in range(n_blocks):
        h_in = states[l]
        h_out = states[l + 1]
        delta = h_out - h_in
        input_norm = torch.linalg.vector_norm(h_in).item()
        output_norm = torch.linalg.vector_norm(h_out).item()
        delta_norm = torch.linalg.vector_norm(delta).item()
        dot = torch.sum(h_in * delta).item()
        cosine = dot / (input_norm * delta_norm) if input_norm > 0 and delta_norm > 0 else float("nan")
        rows.append({
            "model_key": model_key,
            "model_id": MODEL_IDS[model_key],
            "text_id": str(manifest_row["text_id"]),
            "topic_id": str(manifest_row["topic_id"]),
            "topic_domain": str(manifest_row["topic_domain"]),
            "length_tier": str(manifest_row["length_tier"]),
            "style": str(manifest_row["style"]),
            "layer_idx": int(l),
            "n_layers": int(n_blocks),
            "rho_block": l / max(n_blocks - 1, 1),
            "n_tokens": n_tokens,
            "hidden_dim": int(h_in.shape[-1]),
            "input_norm_fro": input_norm,
            "output_norm_fro": output_norm,
            "update_norm_fro": delta_norm,
            "update_norm_per_token": delta_norm / math.sqrt(max(n_tokens, 1)),
            "update_rms_per_element": delta_norm / math.sqrt(max(n_tokens * h_in.shape[-1], 1)),
            "input_update_cosine": cosine,
            "output_input_norm_ratio": output_norm / max(input_norm, 1e-12),
        })
    meta = {
        "n_tokens_runtime": n_tokens,
        "architecture": architecture,
        "n_layers": n_blocks,
        "hidden_dim": int(states[0].shape[-1]),
        "dtype": str(next(model.parameters()).dtype),
        "model_commit_hash": getattr(model.config, "_commit_hash", None),
        "tokenizer_id": getattr(tokenizer, "name_or_path", None),
    }
    return rows, meta


## 5. Run / resume capture

This cell cannot run unless `CAPTURE_ALLOWED` is true. It processes one model at a time and checkpoints metrics/status every five text–model jobs. If the runtime stops, rerun the notebook from the beginning; successful jobs matching the same manifest hash are skipped. A run is marked complete only when every one of the 400 texts succeeds for both models.


In [ ]:
if not CAPTURE_ALLOWED:
    raise RuntimeError("Activation capture blocked by dataset integrity / review gate.")

RUN_DIR = Path("/content") / f"n02_activation_capture_{manifest_file_sha256[:10]}"
RUN_DIR.mkdir(parents=True, exist_ok=True)
METRICS_PATH = RUN_DIR / "N02_activation_metrics.csv"
STATUS_PATH = RUN_DIR / "N02_capture_status.csv"
META_PATH = RUN_DIR / "N02_capture_run_metadata.json"

if META_PATH.exists():
    existing_meta = json.loads(META_PATH.read_text(encoding="utf-8"))
    if existing_meta.get("manifest_file_sha256") != manifest_file_sha256:
        raise ValueError("Existing capture folder belongs to a different manifest hash.")
else:
    existing_meta = {
        "protocol": "N02_Main_Evaluation_Activation_Capture_v1.0",
        "started_utc": datetime.now(timezone.utc).isoformat(),
        "manifest_path": str(manifest_path),
        "manifest_file_sha256": manifest_file_sha256,
        "text_fingerprint_sha256": text_fingerprint_sha256,
        "models_requested": MODEL_IDS,
        "expected_text_model_jobs": int(len(manifest) * len(MODEL_IDS)),
        "python": sys.version.split()[0],
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "device": str(DEVICE),
        "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        "capture_unit": "text_id × model × Transformer block",
        "saved_tensors": "scalar observables only; full block states are transient",
        "status": "IN_PROGRESS",
    }
    META_PATH.write_text(json.dumps(existing_meta, ensure_ascii=False, indent=2), encoding="utf-8")

metrics_df = pd.read_csv(METRICS_PATH) if METRICS_PATH.exists() else pd.DataFrame()
status_df = pd.read_csv(STATUS_PATH) if STATUS_PATH.exists() else pd.DataFrame(
    columns=["model_key", "text_id", "status", "n_layers", "n_tokens", "error", "completed_utc"]
)
if not status_df.empty:
    completed_jobs = set(
        zip(status_df.loc[status_df["status"] == "COMPLETE", "model_key"].astype(str),
            status_df.loc[status_df["status"] == "COMPLETE", "text_id"].astype(str))
    )
else:
    completed_jobs = set()

status_records = status_df.to_dict("records")
status_map = {(str(r["model_key"]), str(r["text_id"])): r for r in status_records}
new_since_checkpoint = 0

for model_key, model_id in MODEL_IDS.items():
    jobs = [(idx, row) for idx, row in manifest.iterrows()
            if (model_key, str(row["text_id"])) not in completed_jobs]
    if not jobs:
        print(f"{model_key}: all text jobs already complete; skip")
        continue

    print(f"Loading {model_id}; remaining jobs: {len(jobs)}")
    tokenizer = model = None
    try:
        tokenizer, model = load_causal_lm(model_id, DTYPE)
        print("Loaded:", model_id, "layers:", len(resolve_blocks(model)[0]), "device:", DEVICE)
        for _, row in jobs:
            text_id = str(row["text_id"])
            job_key = (model_key, text_id)
            try:
                rows, meta = extract_metrics_one_text(tokenizer, model, model_key, row)
                # Resume-safe upsert: if metrics were saved immediately before a runtime stop,
                # replace this text-model pair instead of duplicating its block rows.
                if not metrics_df.empty:
                    keep = ~(
                        metrics_df["model_key"].astype(str).eq(model_key)
                        & metrics_df["text_id"].astype(str).eq(text_id)
                    )
                    metrics_df = metrics_df.loc[keep].copy()
                metrics_df = pd.concat([metrics_df, pd.DataFrame(rows)], ignore_index=True)
                status_map[job_key] = {
                    "model_key": model_key, "text_id": text_id, "status": "COMPLETE",
                    "n_layers": meta["n_layers"], "n_tokens": meta["n_tokens_runtime"],
                    "error": "", "completed_utc": datetime.now(timezone.utc).isoformat()
                }
            except Exception as exc:
                status_map[job_key] = {
                    "model_key": model_key, "text_id": text_id, "status": "ERROR",
                    "n_layers": "", "n_tokens": "", "error": repr(exc),
                    "completed_utc": datetime.now(timezone.utc).isoformat()
                }
                print("ERROR", model_key, text_id, repr(exc))
            new_since_checkpoint += 1
            if new_since_checkpoint >= 5:
                metrics_df.to_csv(METRICS_PATH, index=False)
                pd.DataFrame(list(status_map.values())).to_csv(STATUS_PATH, index=False)
                new_since_checkpoint = 0
                print("Checkpoint:", sum(1 for v in status_map.values() if v["status"] == "COMPLETE"), "completed jobs")
    finally:
        if model is not None:
            del model
        if tokenizer is not None:
            del tokenizer
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

# Final checkpoint
metrics_df.to_csv(METRICS_PATH, index=False)
status_df = pd.DataFrame(list(status_map.values()))
status_df.to_csv(STATUS_PATH, index=False)

expected_jobs = {(key, str(text_id)) for key in MODEL_IDS for text_id in manifest["text_id"].astype(str)}
completed_jobs = set(zip(
    status_df.loc[status_df["status"] == "COMPLETE", "model_key"].astype(str),
    status_df.loc[status_df["status"] == "COMPLETE", "text_id"].astype(str)
))
missing_jobs = expected_jobs - completed_jobs
error_count = int((status_df["status"] == "ERROR").sum())
expected_metric_rows = sum(
    int(len(resolve_blocks_count)) for resolve_blocks_count in []
)  # Report expected row count below from captured per-job layer counts.

complete = (len(missing_jobs) == 0 and error_count == 0)
existing_meta.update({
    "finished_utc": datetime.now(timezone.utc).isoformat(),
    "completed_text_model_jobs": int(len(completed_jobs)),
    "missing_text_model_jobs": int(len(missing_jobs)),
    "error_jobs": error_count,
    "metric_rows": int(len(metrics_df)),
    "status": "COMPLETE" if complete else "INCOMPLETE",
    "metrics_csv": str(METRICS_PATH),
    "status_csv": str(STATUS_PATH),
})
META_PATH.write_text(json.dumps(existing_meta, ensure_ascii=False, indent=2), encoding="utf-8")

print("CAPTURE STATUS:", existing_meta["status"])
print("Completed text×model jobs:", len(completed_jobs), "/", len(expected_jobs))
print("Errors:", error_count, "| Metric rows:", len(metrics_df))
print("Output directory:", RUN_DIR)


## 6. Final validation and export

This cell checks that the completed output contains exactly one row per expected `text_id × model_key × layer_idx`, no duplicate metric keys, and only finite primary observables. If the capture is incomplete, the archive is still saved for debugging but is not labeled a completed run.


In [ ]:
if not METRICS_PATH.exists() or not STATUS_PATH.exists() or not META_PATH.exists():
    raise FileNotFoundError("Capture outputs are missing; run the capture cell first.")

metrics_df = pd.read_csv(METRICS_PATH)
status_df = pd.read_csv(STATUS_PATH)
run_meta = json.loads(META_PATH.read_text(encoding="utf-8"))

key_cols = ["model_key", "text_id", "layer_idx"]
metrics_unique = not metrics_df.duplicated(key_cols).any()
finite_cols = [
    "input_norm_fro", "output_norm_fro", "update_norm_fro",
    "update_norm_per_token", "update_rms_per_element", "input_update_cosine", "output_input_norm_ratio"
]
finite_values = bool(torch.isfinite(torch.tensor(metrics_df[finite_cols].to_numpy(dtype="float64"))).all()) if len(metrics_df) else False
all_jobs_complete = bool((status_df["status"] == "COMPLETE").all()) and len(status_df) == len(manifest) * len(MODEL_IDS)
expected_metric_rows = int(pd.to_numeric(status_df.loc[status_df["status"] == "COMPLETE", "n_layers"], errors="coerce").sum())
metric_row_count_matches_status = len(metrics_df) == expected_metric_rows
if not metrics_df.empty:
    per_job_metric_counts = metrics_df.groupby(["model_key", "text_id"]).size()
    expected_counts = status_df.loc[status_df["status"] == "COMPLETE"].set_index(["model_key", "text_id"])["n_layers"].astype(int)
    metric_layer_coverage_matches = bool(per_job_metric_counts.reindex(expected_counts.index).eq(expected_counts).all()) and len(per_job_metric_counts) == len(expected_counts)
else:
    metric_layer_coverage_matches = False

final_checks = {
    "manifest_hash_recorded": bool(run_meta.get("manifest_file_sha256")),
    "no_duplicate_metric_keys": bool(metrics_unique),
    "primary_numeric_metrics_finite": bool(finite_values),
    "all_800_text_model_jobs_complete": bool(all_jobs_complete),
    "metric_row_count_matches_completed_layer_counts": bool(metric_row_count_matches_status),
    "every_text_model_has_expected_layer_rows": bool(metric_layer_coverage_matches),
    "preflight_report_hash_still_matches_input": hashlib.sha256(manifest_path.read_bytes()).hexdigest() == report.get("frozen_manifest_file_sha256"),
}
display(pd.DataFrame([{"check": k, "pass": v} for k, v in final_checks.items()]))

run_meta["final_validation"] = final_checks
run_meta["status"] = "COMPLETE" if all(final_checks.values()) else "INCOMPLETE"
META_PATH.write_text(json.dumps(run_meta, ensure_ascii=False, indent=2), encoding="utf-8")

archive = Path("/content") / f"N02_Activation_Capture_{manifest_file_sha256[:10]}.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(METRICS_PATH, METRICS_PATH.name)
    z.write(STATUS_PATH, STATUS_PATH.name)
    z.write(META_PATH, META_PATH.name)
    z.write(report_path, "source_preflight_report.json")

print("Run status:", run_meta["status"])
print("Archive:", archive)
from google.colab import files
files.download(str(archive))


## Interpretation guardrails

- The 400 texts form 100 matched topic clusters, not 400 independent units for style contrasts.
- For style effects, resample `topic_id` clusters (10,000 bootstrap replicates), keeping all four style versions together.
- This notebook's output is an activation-derived **metric table**, not the raw `[batch, token, hidden]` tensor archive.
- Do not calculate token-level inferential confidence intervals as though tokens were independent texts.
- Report results only after the run metadata status is `COMPLETE` and the exact manifest hash is preserved.
